In [ ]:
import spacy
import re
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

C:\Users\HP\AppData\Local\Temp\ipykernel_7972\294047344.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [2]:
data = open('data.txt', 'r').read()

In [3]:
data = data.lower()

In [4]:
data = re.sub('\d+\.',"", data)
data = re.sub(r"\b\d+(?:\.\d+)*\.?\b", "", data)

In [5]:
data = re.sub('[^0-9a-zA-Z\s]', "", data).strip()

In [6]:
data = re.sub("\s+", " ", data).strip()

In [7]:
nlp = spacy.load('en_core_web_sm')

In [8]:
tokens = nlp(data)
lemmatize_tokens = [token.lemma_ for token in tokens if not token.is_stop]
data = " ".join(lemmatize_tokens).strip()
data

'artificial intelligence ai machine learn ml deep learning dl complete guide table content introduction artificial intelligence ai history evolution ai type artificial intelligence machine learn ml machine learn work type machine learn supervise learning unsupervised learning semisupervise learn reinforcement learn common machine learning algorithm deep learning dl deep learning work neural network architecture difference ai ml dl application ai ml dl popular tool framework challenge limitation ethic responsible ai future trend conclusion glossary term introduction artificial intelligence machine learning deep learning talkedabout technology twentyfirst century reshape industry change business operate influence everyday life way people notice voice assistant like siri alexa recommendation system netflix amazon selfdrive car medical diagnosis tool technology term interchangeably casual conversation thing artificial intelligence broad concept machine learning subset ai deep learning subs

In [9]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200, 
    chunk_overlap=40, 
    )
data = text_splitter.create_documents([data])


In [10]:
for i in data:
    print(i.page_content)

artificial intelligence ai machine learn ml deep learning dl complete guide table content introduction artificial intelligence ai history evolution ai type artificial intelligence machine learn ml
intelligence machine learn ml machine learn work type machine learn supervise learning unsupervised learning semisupervise learn reinforcement learn common machine learning algorithm deep learning dl
learning algorithm deep learning dl deep learning work neural network architecture difference ai ml dl application ai ml dl popular tool framework challenge limitation ethic responsible ai future
limitation ethic responsible ai future trend conclusion glossary term introduction artificial intelligence machine learning deep learning talkedabout technology twentyfirst century reshape industry
twentyfirst century reshape industry change business operate influence everyday life way people notice voice assistant like siri alexa recommendation system netflix amazon selfdrive car medical
netflix amazon 

In [11]:
ground_truth = {
    "Explain GenAi ?" : [
        "ai agent autonomous system capable perform complex multistep task minimal human intervention explainable ai xai development technique ai decisionmake transparent interpretable ai scientific discovery",
        "ensure ai develop deploy responsibly future trend generative ai continued advancement model capable generate text image video music code multimodal ai system process generate multiple type datum text",
        "generative ai era 2020s emergence large language model llm generative ai tool capable produce humanlike text image audio video mark new chapter ai history bring ai mainstream use million people",
        "function measure difference predict actual value machine learn ml subset ai focus algorithm learn datum model output machine learn algorithm train datum neural network computing system inspire human",
        "network computing system inspire human brain consist interconnect node overfitte model perform training datum poorly new datum regression supervised learning task involve predict continuous value"
    ]
}

In [12]:
embeddings_model = HuggingFaceEmbeddings(
    model_name = 'sentence-transformers/all-miniLM-L6-V2'
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [13]:
vectordb = FAISS.from_documents(documents=data, embedding=embeddings_model)
vectordb

In [14]:
def retrieve(query: str, k: int = 5):
    r_chunks = vectordb.similarity_search(query)
    r_chunks = [doc.page_content for doc in r_chunks]
    retrieved = []
    for i in r_chunks:
        retrieved.append(i)

    return retrieved

In [15]:
def precision_at_k(retrieved: list, relevant: list, k: int) -> float:
    """
    Precision@k = (# relevant docs in top-k) / k
    How many of our top-k results were actually relevant?
    """
    retrieved_k = set(retrieved[:k])
    relevant_set = set(relevant)
    hits = retrieved_k & relevant_set
    return len(hits) / k

In [16]:
def recall_at_k(retrieved: list, relevant: list, k: int) -> float:
    """
    Recall@k = (# relevant docs in top-k) / (# total relevant docs)
    How many of the relevant docs did we actually find?
    """
    retrieved_k = set(retrieved[:k])
    relevant_set = set(relevant)
    hits = retrieved_k & relevant_set
    return len(hits) / len(relevant_set) if relevant_set else 0.0

In [17]:
def reciprocal_rank(retrieved: list, relevant: list) -> float:
    """
    Reciprocal Rank = 1 / (rank of first relevant doc)
    How quickly did we find the FIRST relevant document?
    """
    relevant_set = set(relevant)
    for rank, doc_id in enumerate(retrieved, start=1):
        if doc_id in relevant_set:
            return 1.0 / rank
    return 0.0  # No relevant doc found

In [ ]:
for query, relevant_docs in ground_truth.items():
    retrieved_docs = retrieve(query, k=5)
    
    prec = precision_at_k(retrieved_docs, relevant_docs, 5)
    rec = recall_at_k(retrieved_docs, relevant_docs, 5)
    rank = reciprocal_rank(retrieved_docs, relevant_docs)

print("Precision Score : ",prec)
print("Recall Score : ",rec)
print("Reciprocal Score : ",rank)

Precision Score :  0.6
Recall Score :  0.6
Reciprocal Score :  1.0
